# Day 4 ML/RL Prep

5 basic mixed questions.

Instructions:
- Answer with reasoning, not just the final result.
- Keep answers short and precise.
- If you use code, explain what the tensor means.


## Question 1: Flattening logits for loss

You have policy logits with shape `(B, T, A) = (2, 3, 4)`.
You also have integer actions with shape `(B, T) = (2, 3)`.

If you want to compute cross-entropy over all time steps in one call,
what shapes should the logits and actions be reshaped to?
Why?


In [28]:
# Question 1: Answer and reasoning
import torch
import torch.nn.functional as F

logits = torch.randn(2, 3, 4)
targets = torch.randint(0, 4, (2, 3))

logits_flat = logits.reshape(-1, 4)
targets_flat = targets.reshape(-1)

print('logits shape: ', logits_flat.shape)
print('targets shape: ', targets_flat.shape)

loss = F.cross_entropy(logits_flat, targets_flat)


# Torch's cross entropy requires the logits be specific shape: (N, C), N=batch, C=classes.
# In this example we're given logits with three dimentions, not two.
# If you want to compute the entire loss in one step the batch and time dimentions in both the 
# logtis and target tensors need to be flattened. See code above or (B*T) & (B*T,A).
# The resulting shapes after the flatten before the cross entropy call will be:
# Logits shape: (6, 4). 6 in the batch*time dimention, actions space of 4 containing the networks raw scores for each possible action.
# Targets shape: (6,). 6 in the batch*time dimention, where each of the integer values represent an index within the range of possible actions. 

# Correction: Use the language, expanded accross the feature dimention. Rather than repeated N times. 

logits shape:  torch.Size([6, 4])
targets shape:  torch.Size([6])


## Question 2: Indexing a flattened batch

You have `x` with shape `(B, T, D) = (2, 3, 5)`.
You reshape it with `x_flat = x.reshape(-1, 5)`.

What shape is `x_flat`?
What does one row of `x_flat` represent?


In [39]:
# Question 2: Answer and reasoning

x_shape = torch.randn(2,3,5).reshape(-1, 5).shape
print(x_shape)

# When x is flattened with (-1, 5), we effectivly compute: x_flat = (B*T, D) or (2*3, 5).
# But more technically we compute how many values there are in the whole tensor: 2*3*5=30. 
# With that total value of 30, we then compute 30/5, because we requested 5 in the right dimension. (-1, 5)
# the -1 here becomes 30/5=6.

# Therefore the shape of x_flat is (6,5). 
# One row of the x_flat represents one (batch*time) position turned into a 5d vector. 

torch.Size([6, 5])


## Question 3: Mask broadcasting

You have:
- `x`: shape `(B, T, D) = (2, 3, 5)`
- `mask`: shape `(B, T, 1) = (2, 3, 1)`

Will `x * mask` work directly?
Explain the dimension comparison from the right.


In [32]:
# Question 3: Answer and reasoning
x = torch.randn(2, 3, 5)
mask = torch.randn(2, 3, 1)

print((x*mask).shape)

# The opperation of broadcasting works from right to left. 
# When dimentions are checked, there are two success conditions:
    # Either the dimentions directly match or,
    # One of the dimentions are a 1. 
# In this example all the dimentions macth, apart from the right most. 
# In the rightmost, the mask has a dimention of 1, therefore this can be computed. 
# Why this works: 
    # In the case the dimention is a 1, the value gets repeated D times. In this case D=5.

torch.Size([2, 3, 5])


## Question 4: Small implementation idea

You have:
- `x`: shape `(B, T)`
- `mask`: shape `(B, T)` with 0/1 values

You want a masked mean over all entries where `mask == 1`.

Do not write full polished code unless you want to.
I want:
- the formula
- what the numerator means
- what the denominator means
- one edge case you would handle


In [156]:
# Question 4: Answer and reasoning
x = torch.randn(2, 4)
mask = torch.randint(0, 2, (2, 4))

def zero_mask_avg(x: torch.tensor, mask: torch.tensor) -> int:
    y = x*mask
    return y.sum() / ((y !=0).sum().item() + 1e-9)

print('Zero mask Average of x: ', zero_mask_avg(x, mask))

# The formula: \bar{y} = \frac{\sum_{i=1}^{n} (x_i \cdot m_i)}{\sum_{i=1}^{n} \mathbb{1}_{(x_i \cdot m_i \neq 0)}}
# The numerator means: sum off all values in y, where y=x*mask.
# The denominator means: The count of all non-0 values of y, where y=x*mask.
# Here to calulate the average we must divide. And division by 0 doesn't compute. 
# To avoid this, we could add a very small value to the denominator to avoid division by 0 error. 

# this is slightly buggy but right idea. 

Zero mask Average of x:  tensor(0.1953)


## Question 5: Debugging policy gradient shapes

Consider:

```python
logits = policy(obs)          # (B, T, A)
dist = torch.distributions.Categorical(logits=logits)
logp = dist.log_prob(actions) # (B, T)
loss = -(logp * returns).mean()
```

Suppose `returns` has shape `(B,)`.

What is the problem?
What shape should `returns` have if the loss is per time step?


In [192]:
# Question 5: Answer and reasoning

B = 2
T = 3
A = 4

logits = torch.randn(B, T, A)
actions = torch.randint(0, 4, (B, T))
returns = torch.randint(0, 10, (B, ))

print(logits.shape)
print(actions.shape)
print(returns.shape)

dist = torch.distributions.Categorical(logits=logits)
logp = dist.log_prob(actions)
loss = -(logp * returns).mean()

# (logp * returns) cannot be computed as the shapes are logp(B,T) returns(B).
# Here the returns are the total reward for entire trajectories, where each trajectory is a batch.
# If the loss is per timestep, returns may have be split into rewards per timestep and have shape (B,T) matching logp.
# However this is impossible to calulate exactly in retrospect. 
# If we wanted to solve this problem in retrospect we could sum the logits accross the A dimention. Therefore calculating the probabilites of entire trajectories.
# Here logp would be shape (B,), so (logp * returns) can be computed. 
# However, this is not advised, instead the prior approach should be taken.

torch.Size([2, 3, 4])
torch.Size([2, 3])
torch.Size([2])


RuntimeError: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1